# 07 — Clustering Analysis: Alternative Borough Prioritisation

The composite ranking in `05_build_gold.ipynb` uses three equally-weighted dimensions:
- % social stock below EPC C
- Income deprivation rate (IMD 2019)
- % pre-1950 stock

**Problem identified**: pre-1950 stock and EPC quality are correlated — old buildings tend to have worse EPCs. This means a borough with lots of old stock gets penalised twice, inflating its rank. Hammersmith and Fulham is the clearest case: it ranks 4th overall largely because of pre-1950 stock (rank 1), but its EPC quality rank is 10 and income rank is 14.

**Fix applied here**: we replace % pre-1950 with **% of wall insulation recommendations that are solid wall (improvement ID 7)** as the third dimension. This is a direct retrofit cost signal from actual surveyor assessments — not a proxy based on age. Solid wall insulation costs £8,000–£25,000 per property vs ~£1,500 for cavity wall, so this metric captures genuine retrofit difficulty without double-counting the EPC quality signal.

The notebook then explores three alternative approaches that handle correlated features more robustly:

1. **Correlation check** — confirm the new feature set is less correlated
2. **PCA** (Principal Component Analysis) — mathematically separates correlated features into independent components
3. **K-means clustering** — groups boroughs by similarity without requiring weights; elbow method to find optimal k
4. **Hierarchical clustering** — builds a tree of borough similarities; no need to specify k upfront

At the end we compare all outputs side-by-side and pick the best approach for the final portfolio summary.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage

GOLD = '../data/gold'

# Load borough priority (from 05_build_gold.ipynb)
priority = pd.read_parquet(f'{GOLD}/borough_priority')

# Load wall type analysis (from 06_recommendations_analysis.ipynb)
# pct_solid_wall = % of wall insulation recs that are solid wall (ID 7)
# This replaces % pre-1950 as the retrofit difficulty dimension
wall = pd.read_parquet(f'{GOLD}/borough_wall_type')[['borough', 'pct_solid_wall']]

df = priority.merge(wall, on='borough', how='left')
df = df.sort_values('priority_rank').reset_index(drop=True)

print(f'Boroughs loaded: {len(df)}')
print(f'Boroughs with solid wall data: {df["pct_solid_wall"].notna().sum()}')
print()
print(df[['borough','pct_below_epc_c','income_deprivation_rate',
          'pct_pre_1950','pct_solid_wall','priority_rank']].to_string())

## 1. Correlation check — how bad is the double-counting?

Before fixing a problem, quantify it. If EPC quality and pre-1950 stock have a strong correlation, the existing ranking is giving too much weight to housing age. We look at the Pearson correlation matrix and a scatter plot.

In [ ]:
features = ['pct_below_epc_c', 'income_deprivation_rate', 'pct_solid_wall']
labels   = ['% below EPC C', 'Income deprivation', '% solid wall recs']

# Drop any boroughs missing solid wall data and reset index so it aligns with numpy arrays
df_clean = df.dropna(subset=features).reset_index(drop=True)
print(f'Boroughs used in analysis: {len(df_clean)} (dropped {len(df)-len(df_clean)} missing solid wall data)')

corr = df_clean[features].corr()
corr.index   = labels
corr.columns = labels

print()
print('=== Pearson correlation matrix (NEW features) ===')
print(corr.round(3))
print()
print('Compare: original pct_below_epc_c vs pct_pre_1950 correlation was ~0.4-0.5')
print('New: pct_below_epc_c vs pct_solid_wall should be lower — less double-counting')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
            vmin=-1, vmax=1, ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Feature correlation matrix (updated features)', fontweight='bold')

axes[1].scatter(df_clean['pct_solid_wall'], df_clean['pct_below_epc_c'], alpha=0.7, s=60, color='steelblue')
for _, row in df_clean.iterrows():
    axes[1].annotate(row['borough'][:10], (row['pct_solid_wall'], row['pct_below_epc_c']),
                     fontsize=6, alpha=0.7, xytext=(3,2), textcoords='offset points')
r = df_clean['pct_below_epc_c'].corr(df_clean['pct_solid_wall'])
m, b = np.polyfit(df_clean['pct_solid_wall'], df_clean['pct_below_epc_c'], 1)
x_line = np.linspace(df_clean['pct_solid_wall'].min(), df_clean['pct_solid_wall'].max(), 100)
axes[1].plot(x_line, m*x_line+b, color='red', linestyle='--', alpha=0.6, label=f'r = {r:.2f}')
axes[1].set_xlabel('% solid wall recommendations')
axes[1].set_ylabel('% below EPC C')
axes[1].set_title('EPC quality vs solid wall prevalence', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/07_correlation_check.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature preparation — normalise before any ML

All three clustering/PCA approaches require features on the same scale. `% below EPC C` ranges 32–56; `income deprivation` ranges 0.06–0.20. Without normalising, EPC would dominate simply because its numbers are larger.

We use `StandardScaler`: subtract the mean, divide by standard deviation. Each feature ends up with mean=0, std=1.

In [ ]:
X = df_clean[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Before scaling (first 3 rows):')
print(pd.DataFrame(X[:3], columns=labels).round(3))
print()
print('After scaling (first 3 rows):')
print(pd.DataFrame(X_scaled[:3], columns=labels).round(3))
print()
print('Scaled means:', X_scaled.mean(axis=0).round(5))
print('Scaled stds: ', X_scaled.std(axis=0).round(5))

## 3. PCA — Principal Component Analysis

**What it does**: PCA finds new axes (called principal components) that are:
- Aligned with the directions of maximum variance in the data
- Guaranteed to be uncorrelated with each other

In our case, if EPC quality and pre-1950 are correlated, PCA will combine them into one component rather than treating them as two separate signals.

**How to read the output**:
- **Explained variance ratio**: how much of the total variation in the data each component captures
- **Loadings**: how much each original feature contributes to each component (positive = same direction, negative = opposite)
- PC1 is the most important component — it captures the most variation. We can rank boroughs on PC1 as an alternative to our composite score.

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

# Explained variance
print('=== Explained variance per component ===')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.1%}  (cumulative: {pca.explained_variance_ratio_[:i+1].sum():.1%})')

# Loadings
print()
print('=== Component loadings (contribution of each feature) ===')
loadings = pd.DataFrame(pca.components_.T, index=labels,
                         columns=['PC1', 'PC2', 'PC3'])
print(loadings.round(3))
print()
print('Interpretation: large absolute value = strong contribution.')
print('Same sign = features move together. Opposite sign = features trade off.')

In [ ]:
# Rank boroughs on PC1 (higher PC1 score = worse on the dominant dimension)
df_clean['pc1_score'] = X_pca[:, 0]
df_clean['pc2_score'] = X_pca[:, 1]

# Flip sign if needed so that high PC1 = high priority (check vs known top boroughs)
if df_clean[df_clean['borough'] == 'Barking and Dagenham']['pc1_score'].values[0] < 0:
    df_clean['pc1_score'] = -df_clean['pc1_score']
    X_pca[:, 0] = -X_pca[:, 0]

df_clean['pca_rank'] = df_clean['pc1_score'].rank(ascending=False).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs PC2 scatter, coloured by original priority rank
sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df_clean['priority_rank'], cmap='RdYlGn', s=80, alpha=0.8)
for idx, row in df_clean.iterrows():
    axes[0].annotate(row['borough'][:8], (X_pca[idx, 0], X_pca[idx, 1]),
                     fontsize=6, alpha=0.8, xytext=(3, 2), textcoords='offset points')
plt.colorbar(sc, ax=axes[0], label='Original priority rank')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)')
axes[0].set_title('Boroughs in PCA space', fontweight='bold')
axes[0].axhline(0, color='grey', linestyle='--', alpha=0.3)
axes[0].axvline(0, color='grey', linestyle='--', alpha=0.3)

# Original rank vs PCA rank
axes[1].scatter(df_clean['priority_rank'], df_clean['pca_rank'], s=60, alpha=0.7, color='steelblue')
for _, row in df_clean.iterrows():
    if abs(row['priority_rank'] - row['pca_rank']) >= 4:
        axes[1].annotate(row['borough'][:10], (row['priority_rank'], row['pca_rank']),
                         fontsize=7, color='red', xytext=(3, 2), textcoords='offset points')
axes[1].plot([1, 33], [1, 33], 'r--', alpha=0.4, label='No change')
axes[1].set_xlabel('Original composite rank')
axes[1].set_ylabel('PCA rank (PC1)')
axes[1].set_title('Where boroughs move between methods', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/07_pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Boroughs that move most between original rank and PCA rank ===')
df_clean['rank_shift_pca'] = df_clean['pca_rank'] - df_clean['priority_rank']
print(df_clean[['borough','priority_rank','pca_rank','rank_shift_pca']]
      .sort_values('rank_shift_pca', key=abs, ascending=False).head(10).to_string(index=False))

## 4. K-means clustering — finding natural groups

**What it does**: groups boroughs into k clusters so that boroughs within the same cluster are as similar as possible to each other, and as different as possible from other clusters. No weights required.

**How we pick k**: two methods:
- **Elbow method**: plot within-cluster sum of squares (inertia) vs k. The elbow point where improvement flattens is the best k.
- **Silhouette score**: measures how well each borough fits its cluster vs the next-closest cluster. Range -1 to 1; higher = better-defined clusters.

We test k=2 through k=7.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow method', fontweight='bold')
axes[0].set_xticks(list(k_range))
for k, v in zip(k_range, inertias):
    axes[0].annotate(f'k={k}', (k, v), textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[1].plot(k_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score (higher = better)', fontweight='bold')
axes[1].set_xticks(list(k_range))
for k, v in zip(k_range, silhouettes):
    axes[1].annotate(f'{v:.2f}', (k, v), textcoords='offset points', xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/07_kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f'Best k by silhouette score: {best_k}')
print(f'Silhouette scores: {dict(zip(k_range, [round(s,3) for s in silhouettes]))}')

In [ ]:
# Fit at best k AND also test k=3 for interpretability comparison
for k in sorted(set([best_k, 3])):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_scaled)

    # Order clusters by mean PC1 score so cluster 0 = highest priority
    cluster_pc1 = {c: X_pca[cluster_labels == c, 0].mean() for c in range(k)}
    rank_order = sorted(cluster_pc1, key=cluster_pc1.get, reverse=True)
    remap = {old: new for new, old in enumerate(rank_order)}
    cluster_ordered = np.array([remap[c] for c in cluster_labels])

    tier_labels = {0: 'High priority', 1: 'Medium priority', 2: 'Lower priority',
                   3: 'Lowest priority', 4: 'Lowest priority'}

    df_clean[f'kmeans_k{k}'] = cluster_ordered
    df_clean[f'kmeans_k{k}_label'] = [tier_labels.get(c, f'Cluster {c}') for c in cluster_ordered]

    print(f'\n=== K-means k={k} cluster assignments ===')
    for c in sorted(set(cluster_ordered)):
        boroughs_in = df_clean[df_clean[f'kmeans_k{k}'] == c]['borough'].tolist()
        mean_epc = df_clean[df_clean[f'kmeans_k{k}'] == c]['pct_below_epc_c'].mean()
        mean_inc = df_clean[df_clean[f'kmeans_k{k}'] == c]['income_deprivation_rate'].mean()
        mean_age = df_clean[df_clean[f'kmeans_k{k}'] == c]['pct_pre_1950'].mean()
        print(f'  {tier_labels.get(c, f"Cluster {c}")} ({len(boroughs_in)} boroughs):')
        print(f'    Avg EPC below C: {mean_epc:.1f}%  |  Avg income deprivation: {mean_inc:.3f}  |  Avg pre-1950: {mean_age:.1f}%')
        print(f'    Boroughs: {", ".join(boroughs_in)}')

In [ ]:
# Silhouette plot for best k — shows how well each borough fits its cluster
fig, ax = plt.subplots(figsize=(10, 6))

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_best = km_best.fit_predict(X_scaled)
sil_vals = silhouette_samples(X_scaled, labels_best)

y_lower = 10
colours = plt.cm.tab10(np.linspace(0, 1, best_k))

for c in range(best_k):
    c_sil = np.sort(sil_vals[labels_best == c])
    size = c_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     facecolor=colours[c], alpha=0.7, label=f'Cluster {c}')
    ax.text(-0.05, y_lower + 0.5 * size, str(c))
    y_lower = y_upper + 5

ax.axvline(x=silhouette_score(X_scaled, labels_best), color='red', linestyle='--',
           label=f'Mean silhouette = {silhouette_score(X_scaled, labels_best):.2f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Borough (grouped by cluster)')
ax.set_title(f'Silhouette plot — K-means k={best_k}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/07_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bars extending right of the dashed line = well-matched to cluster.')
print('Bars extending left = borderline boroughs that could belong to another cluster.')

## 5. Hierarchical clustering — building the similarity tree

**What it does**: instead of picking k upfront, hierarchical clustering builds a tree (dendrogram) showing how similar every pair of boroughs is. You can then cut the tree at any height to get any number of clusters.

**Linkage method**: we use `ward` linkage, which minimises the total within-cluster variance at each merge step. This tends to produce compact, evenly-sized clusters and is the most commonly used method.

**How to read the dendrogram**: boroughs that merge low on the y-axis are very similar. The y-axis value at which two branches join is the distance between them. A large jump in height suggests a natural cluster boundary there.

In [ ]:
Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dend = dendrogram(Z, labels=df_clean['borough'].tolist(), ax=ax,
                  color_threshold=0.7 * max(Z[:, 2]),
                  leaf_rotation=90, leaf_font_size=8)
ax.set_title('Hierarchical clustering dendrogram (Ward linkage)', fontweight='bold')
ax.set_ylabel('Distance (Ward criterion)')
ax.set_xlabel('Borough')

# Mark cut points for 3 and 4 clusters
for n_cut, colour in [(3, 'red'), (4, 'blue')]:
    cut_height = sorted(Z[:, 2], reverse=True)[n_cut - 2]
    ax.axhline(y=cut_height, color=colour, linestyle='--', alpha=0.6,
               label=f'Cut for {n_cut} clusters')

ax.legend()
plt.tight_layout()
plt.savefig('../outputs/07_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for n_clusters in [3, 4]:
    hc = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    hc_labels = hc.fit_predict(X_scaled)

    # Order so cluster 0 = highest avg PC1
    cluster_pc1 = {c: X_pca[hc_labels == c, 0].mean() for c in range(n_clusters)}
    rank_order = sorted(cluster_pc1, key=cluster_pc1.get, reverse=True)
    remap = {old: new for new, old in enumerate(rank_order)}
    hc_ordered = np.array([remap[c] for c in hc_labels])

    df_clean[f'hc_{n_clusters}'] = hc_ordered

    tier_labels = {0: 'High priority', 1: 'Medium priority', 2: 'Lower priority', 3: 'Lowest priority'}
    print(f'\n=== Hierarchical clustering — {n_clusters} clusters ===')
    for c in sorted(set(hc_ordered)):
        boroughs_in = df_clean[df_clean[f'hc_{n_clusters}'] == c]['borough'].tolist()
        print(f'  {tier_labels[c]} ({len(boroughs_in)}): {", ".join(boroughs_in)}')

## 6. Side-by-side comparison — do the methods agree?

The key question: are the boroughs that appear in the high-priority group consistent across methods? If they are, that's strong evidence those boroughs genuinely need the most attention regardless of methodological choices.

In [ ]:
# Use best_k for k-means label, 3-cluster HC
comparison = df_clean[[
    'borough', 'priority_rank',
    'pca_rank',
    f'kmeans_k{best_k}_label',
    'hc_3',
    'pct_below_epc_c', 'income_deprivation_rate', 'pct_pre_1950'
]].copy()

hc_tier = {0: 'High priority', 1: 'Medium priority', 2: 'Lower priority'}
comparison['hc_3_label'] = comparison['hc_3'].map(hc_tier)

comparison = comparison.drop(columns=['hc_3'])
comparison.columns = [
    'Borough', 'Original rank', 'PCA rank',
    f'K-means (k={best_k})', 'Hierarchical (3 clusters)',
    '% below EPC C', 'Income deprivation', '% pre-1950'
]

print('=== Full comparison ===')
print(comparison.sort_values('Original rank').to_string(index=False))

In [ ]:
print('=== Boroughs with largest rank shift: Original → PCA ===')
df_clean['abs_shift'] = (df_clean['pca_rank'] - df_clean['priority_rank']).abs()
print(df_clean[['borough','priority_rank','pca_rank','pct_below_epc_c','income_deprivation_rate','pct_pre_1950']]
      .sort_values('abs_shift', ascending=False).head(8).to_string(index=False))

print()
print('=== Top 5 high-priority boroughs: are they consistent across all methods? ===')
top5_original = set(df_clean[df_clean['priority_rank'] <= 5]['borough'])
top5_pca      = set(df_clean[df_clean['pca_rank'] <= 5]['borough'])
top_hc        = set(df_clean[df_clean['hc_3'] == 0]['borough'])
top_km        = set(df_clean[df_clean[f'kmeans_k{best_k}'] == 0]['borough'])

print(f'Original top 5:            {sorted(top5_original)}')
print(f'PCA top 5:                 {sorted(top5_pca)}')
print(f'Hierarchical high-priority: {sorted(top_hc)}')
print(f'K-means high-priority:      {sorted(top_km)}')
print()
print(f'Consistent across ALL methods: {sorted(top5_original & top5_pca & top_hc & top_km)}')

## 7. Summary and recommendation

Which method should we use as the primary output?

In [ ]:
print('=' * 65)
print('SUMMARY OF FINDINGS')
print('=' * 65)
print()
print('Correlation between EPC quality and pre-1950 stock:',
      round(df['pct_below_epc_c'].corr(df['pct_pre_1950']), 3))
print()
print('The three alternative methods agree on the core finding:')
consistent = sorted(top5_original & top5_pca & top_hc & top_km)
print(f'  {consistent}')
print('appear in the high-priority group across ALL methods.')
print()
print('Boroughs most affected by the double-counting problem:')
movers = df_clean[df_clean['abs_shift'] >= 4][['borough','priority_rank','pca_rank']].sort_values('abs_shift', ascending=False)
print(movers.to_string(index=False))
print()
print('Recommendation:')
print('  K-means clustering is the most interpretable output for a housing')
print('  association audience. Rather than a precise rank (which implies false')
print('  precision), clustering groups boroughs into tiers: high / medium / lower')
print('  priority. The top cluster is stable regardless of method.')
print()
print('  PCA confirms the correlation between EPC and age accounts for')
print(f'  {pca.explained_variance_ratio_[0]:.0%} of variance in a single component,')
print('  validating the concern about double-counting.')
print()
print('  Hierarchical clustering and the dendrogram are useful for showing')
print('  HOW SIMILAR neighbouring boroughs are — a good visual for a portfolio.')